## Import Libraries

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import PolynomialFeatures,OneHotEncoder,StandardScaler
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split,GridSearchCV
from xgboost import XGBRegressor
import seaborn as sns
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt


## Importing Dataset

In [5]:
df = pd.read_csv("CarPrice_Assignment.csv")
# checking empty values
# df.isnull().sum()
# No empty Values
df["brand"] = df["CarName"].str.split().str[0].str.lower()
df["brand"] = df["brand"].replace({
    "maxda": "mazda", "porcshce": "porsche", "toyouta": "toyota",
    "vokswagen": "volkswagen", "vw": "volkswagen",
})
df = df.drop(columns=["car_ID", "CarName"])
x=df.drop(columns=["price"])
y=df["price"]

## Splitting data

In [6]:
x_train, x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [7]:
def report(y_true,y_pred):
    return print(f"R2 score:{r2_score(y_true,y_pred)}\nRSME:{root_mean_squared_error(y_true,y_pred)}")

## Label Encoding And Standardization

In [8]:
cat_cols = x.select_dtypes(include="object").columns.tolist()
num_cols = x.select_dtypes(exclude="object").columns.tolist()
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

## Multiple Linear regression

In [9]:
mlp= Pipeline([
    ("prep", preprocess),
    ("mlp", LinearRegression()),
])
mlp.fit(x_train,y_train)
mlp_pred= mlp.predict(x_test)

## Polynomial Regeression

In [10]:
num_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

pipe = Pipeline([
    ("prep", preprocess),
    ("reg", Ridge()),
])

# log-transform the skewed target
model = TransformedTargetRegressor(
    regressor=pipe, func=np.log1p, inverse_func=np.expm1
)

# tune degree and regularization strength together
param_grid = {
    "regressor__prep__num__poly__degree": [1, 2, 3],
    "regressor__reg__alpha": [0.1, 1, 10, 100, 1000],
}

search = GridSearchCV(model, param_grid, cv=5, scoring="r2", n_jobs=-1)
search.fit(x_train, y_train)

print("Best params:", search.best_params_)
print("CV R2:", search.best_score_)

poly_pred = search.predict(x_test)

Best params: {'regressor__prep__num__poly__degree': 3, 'regressor__reg__alpha': 1}
CV R2: 0.9208971201050167


## Decision Tree

In [11]:
dt_pipline =Pipeline([("prep",preprocess),("dt",DecisionTreeRegressor())])
param_grid ={"dt__max_depth":[None,10,15,20,30,25]}
dt_search=GridSearchCV(dt_pipline,param_grid,cv=5,scoring="r2",n_jobs=-1)
dt_search.fit(x_train,y_train)
dt_pred=dt_search.predict(x_test)

## Random Forest

In [12]:
rf_pipline=Pipeline([("prep",preprocess),("rf",RandomForestRegressor())])
param_grid={"rf__n_estimators":[100,150,200,350,300]}
rf_search=GridSearchCV(rf_pipline,param_grid,cv=5,scoring="r2",n_jobs=-1)
rf_search.fit(x_train,y_train)
rf_pred=rf_search.predict(x_test)

## XGBoost

In [13]:
xg_pipline=Pipeline([("perp",preprocess),("xg",XGBRegressor(subsample=0.8,colsample_bytree=0.8,objective='reg:squarederror',random_state=0))])
param_grid={'xg__n_estimators':[100,150,200,250,300],"xg__learning_rate":[0.05,0.1,0.15,0.2,0.25],"xg__max_depth":[5,6,7,8,9,10]}
xg_search=GridSearchCV(xg_pipline,param_grid,cv=5,scoring="r2",n_jobs=-1)
xg_search.fit(x_train,y_train)
xg_pred=xg_search.predict(x_test)

## SVM

In [14]:
svm=Pipeline([("prep",preprocess),('svm',SVR(kernel="rbf"))])
svm.fit(x_train,y_train)
svm_pred= svm.predict(x_test)


## Neural Network

In [15]:
prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
Xtr = prep.fit_transform(x_train)
Xte = prep.transform(x_test)

ytr_log = np.log1p(y_train.values)
y_scaler = StandardScaler()
ytr = y_scaler.fit_transform(ytr_log.reshape(-1, 1)).ravel()

tf.random.set_seed(42)
model = keras.Sequential([
    keras.layers.Input(shape=(Xtr.shape[1],)),
    keras.layers.Dense(32, activation="relu",
                       kernel_regularizer=keras.regularizers.l2(1e-2)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation="relu",
                       kernel_regularizer=keras.regularizers.l2(1e-2)),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")

early = keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True)
model.fit(Xtr, ytr, validation_split=0.15, epochs=500,batch_size=16, callbacks=[early], verbose=0)


pred = np.expm1(y_scaler.inverse_transform(model.predict(Xte)).ravel())

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


In [17]:
predictions= [("Multiple Linear Regression",mlp_pred),("Polynomial Regerssion",poly_pred),("Decision Tree",dt_pred),("Random Forest",rf_pred),('SVM',svm_pred),("XGboost",xg_pred),("Neural Network",pred)]
for name, preds in predictions:
    print(f"{name} Report")
    report(y_test, preds)

Multiple Linear Regression Report
R2 score:0.9010854026798552
RSME:2794.4079938592445
Polynomial Regerssion Report
R2 score:0.8781272913602048
RSME:3101.79329671086
Decision Tree Report
R2 score:0.9178898212712192
RSME:2545.998362921173
Random Forest Report
R2 score:0.9547588544077107
RSME:1889.846010196948
SVM Report
R2 score:-0.09972546394037907
RSME:9317.550643125074
XGboost Report
R2 score:0.9422082449656627
RSME:2135.9572833047923
Neural Network Report
R2 score:0.9335755495420622
RSME:2289.9373799017358


In [ ]:
report(y_test,mlp_pred)